# CTDDG · Five-model generation preparation · 10 independent runs

**PyTorch generation for four models:** previous unconditional CDGCN, previous toxicity-weighted CTDDG, new five-epoch clean CDGCN, and new **protein-conditioned** BindingDB fine-tuned CTDDG. The fifth model, `nitesh_model` (MXNet), is **not** generated by this PyTorch notebook; its pre-existing CSVs are audited for completeness.

- **No retraining or checkpoint rewriting.** Checkpoints are read in place; all model loads use `strict=True` with training-specific architectures and vocabularies.
- **Skip complete outputs:** a protein/model with verified finished sample CSVs is left untouched, including when rerunning the notebook.
- **Source fidelity:** the clean CDGCN uses its 55-atom vocabulary, `mask` input alias, and ReLU after the backbone's final dense layers; the old models use their original vocabulary and backbone; the fine-tuned model additionally uses the actual saved protein projection layers and `init_head`.
- **Fair comparison caveat:** the three unconditional PyTorch models sample independent groups for each protein slot. Only the fine-tuned model conditions on its protein vector. The historical MXNet model has a different architecture; do not claim a like-for-like paper reproduction.
- **Outputs:** `REWORKED_CODE/testing/Validity_Analysis/Generated_104_10runs/<model>/proteinN/`. Raw attempts are retained for unbiased validity calculation. Models are **not** silently regenerated when checkpoint identities change.

**Fine-tuning provenance:** the attached fine-tuning notebook starts from the **older** `cdgcn_nll` pretraining checkpoint, and its experiment is tagged `exploratory_leaked_pretraining`. Do not describe it as fine-tuned from the new 5-epoch checkpoint or as a leakage-free benchmark.

## 1 · Paths, exact checkpoint sources, model variants, and safe skipping

In [ ]:
from pathlib import Path
import sys, json, re, math, random, time, gc, hashlib, os
from dataclasses import dataclass

ROOT = Path(os.environ.get('CTDDG_ROOT', '/home/muhamed/repo/CTDDG')).expanduser().resolve()
REWORKED = ROOT / 'REWORKED_CODE'
VALIDITY_ROOT = REWORKED / 'testing' / 'Validity_Analysis'
EMBEDDINGS_FOLDER = VALIDITY_ROOT / 'Embeddings_104' / 'embeddings_104'
MODEL_ROOT = REWORKED / 'model_weights'
OUTPUT_ROOT = VALIDITY_ROOT / 'Generated_104_10runs'

# Existing pretrained runs. New model checkpoints are read directly from the
# original training folders; copying them to model_weights is NOT required.
CLEAN_PRETRAIN_DIR = (REWORKED / 'PRETRAINING_OUTPUT/no_toxicity/training_runs'
                      / 'chembl_B32_K5_alpha08_seed42')
FINETUNE_RUN_DIR = (REWORKED / 'finetuning/finetuning_runs'
                    / 'cdgcn_to_ctddg_d1_bs32_lambda0p1_v1')
LEGACY_ATOM_TYPES = REWORKED / 'chembl_preprocessed/atom_types.txt'
CLEAN_ATOM_TYPES = REWORKED / 'PREPROCESSING_OUTPUT/vocabulary/atom_types.txt'

# The historical MXNet nitesh_model is inspected but not loaded by PyTorch.
ALL_MODEL_NAMES = [
    'nitesh_model',
    'Pretrained_no_toxicity',
    'Pretrained_with_toxicity',
    'Pretrained_clean_5epoch',
    'Finetuned_BindingDB_ctddg',
]
MODEL_NAMES = ALL_MODEL_NAMES[1:]  # The four PyTorch models to generate

MODEL_SOURCES = {
    'Pretrained_no_toxicity': {
        'kind': 'legacy_unconditional', 'vocab': LEGACY_ATOM_TYPES,
        'candidates': [MODEL_ROOT/'Pretrained_no_toxicity/model_weights_final.pt',
                       MODEL_ROOT/'Pretrained_no_toxicity/checkpoint.pt'],
        'metadata_dir': MODEL_ROOT/'Pretrained_no_toxicity',
        'dense_relu': False,
    },
    'Pretrained_with_toxicity': {
        'kind': 'legacy_unconditional', 'vocab': LEGACY_ATOM_TYPES,
        'candidates': [MODEL_ROOT/'Pretrained_with_toxicity/checkpoint.pt'],
        'metadata_dir': MODEL_ROOT/'Pretrained_with_toxicity',
        'dense_relu': False,
    },
    'Pretrained_clean_5epoch': {
        'kind': 'clean_unconditional', 'vocab': CLEAN_ATOM_TYPES,
        'candidates': [CLEAN_PRETRAIN_DIR/'best.pt',
                       MODEL_ROOT/'Pretrained_clean_5epoch/model_weights_final.pt'],
        'metadata_dir': CLEAN_PRETRAIN_DIR,
        'dense_relu': True,
    },
    'Finetuned_BindingDB_ctddg': {
        'kind': 'bindingdb_conditional', 'vocab': LEGACY_ATOM_TYPES,
        'candidates': [FINETUNE_RUN_DIR/'ckpt_best.pt',
                       FINETUNE_RUN_DIR/'ckpt_last.pt'],
        'metadata_dir': FINETUNE_RUN_DIR,
        'dense_relu': False, 'n_c': 6165,
    },
}

TOTAL_RUNS = 10
TARGET_PER_RUN = 100      # unique, RDKit-sanitizable completed molecules per run
BATCH_SIZE = 8
MAX_ACTIONS = 120
MAX_ATOMS = 120
MAX_ATTEMPTS_PER_RUN = 3000
SEED = 42
GREEDY = False
SMOKE_TEST = False    # True: isolated 2-protein, one-run, 10-sample smoke test
SMOKE_PROTEINS = 2
SMOKE_TARGET = 10
SMOKE_MAX_ATTEMPTS = 200
SAVE_RAW_EVERY_BATCHES = 5
SHOW_PROGRESS_EVERY_BATCHES = 5
SKIP_COMPLETE = True
TRUST_LOCAL_CHECKPOINT = True # trusted files produced by your own jobs only

if SMOKE_TEST:
    effective_runs, effective_target = 1, SMOKE_TARGET
    effective_max_attempts = SMOKE_MAX_ATTEMPTS
    SAVE_ROOT = VALIDITY_ROOT / 'smoke_test_five_models' / 'Generated_104_10runs'
else:
    effective_runs, effective_target = TOTAL_RUNS, TARGET_PER_RUN
    effective_max_attempts = MAX_ATTEMPTS_PER_RUN
    SAVE_ROOT = OUTPUT_ROOT

assert len(ALL_MODEL_NAMES) == 5 and len(MODEL_NAMES) == 4
assert BATCH_SIZE >= 1 and MAX_ACTIONS >= 1 and MAX_ATOMS >= 1
assert effective_target > 0 and effective_runs > 0 and effective_max_attempts >= effective_target
print('Project root       :', ROOT)
print('Protein embeddings :', EMBEDDINGS_FOLDER)
print('New pretrain run   :', CLEAN_PRETRAIN_DIR)
print('Fine-tuning run    :', FINETUNE_RUN_DIR)
print('Output folder      :', SAVE_ROOT)
print('Models to generate :', MODEL_NAMES)
print('MODE               :', 'SMOKE' if SMOKE_TEST else 'FULL')


## 2 · Environment and package verification

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import torch
from torch import nn
from torch.nn import functional as F
from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import Draw
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RDLogger.DisableLog('rdApp.error')
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| device:', DEVICE)
print('RDKit:', rdBase.rdkitVersion, '| NumPy:', np.__version__, '| SciPy:', scipy.__version__)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
assert EMBEDDINGS_FOLDER.is_dir(), f'Embedding directory not found: {EMBEDDINGS_FOLDER}'
# Old PyTorch weights are only required for models which need generation.
# Completed model outputs can be skipped even when weights are archived.
assert LEGACY_ATOM_TYPES.is_file(), f'Missing legacy atom vocabulary: {LEGACY_ATOM_TYPES}'
assert CLEAN_ATOM_TYPES.is_file(), f'Missing new 55-atom vocabulary: {CLEAN_ATOM_TYPES}' 

## 3 · Protein embeddings — verify all 104, required for the fine-tuned model

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

files = sorted(
    (p for p in EMBEDDINGS_FOLDER.glob('protein*.txt')
     if re.fullmatch(r'protein[0-9]+\.txt', p.name)),
    key=lambda p: int(p.stem.replace('protein', ''))
)
ids = [int(p.stem[7:]) for p in files]
expected = set(range(1, 105))
if set(ids) != expected or len(files) != 104:
    raise ValueError('Expected exactly protein1.txt ... protein104.txt. '
                     f'Missing: {sorted(expected - set(ids))}; extra: {sorted(set(ids)-expected)}; '
                     f'found {len(files)} files in {EMBEDDINGS_FOLDER}')

emb_rows=[]
protein_vectors = {}
for p in files:
    v = np.loadtxt(p, dtype=np.float32)
    if v.size == 0 or not np.isfinite(v).all():
        raise ValueError(f'Embedding file empty or contains NaN/Inf: {p}')
    if v.ndim not in (1, 2) or (v.ndim==2 and min(v.shape)!=1):
        raise ValueError(f'Unexpected embedding shape for {p.name}: {v.shape}')
    protein_vectors[p.stem] = np.asarray(v, dtype=np.float32).reshape(-1)
    emb_rows.append({'protein_id':p.stem,'embedding_file':str(p),
                     'dimension':int(v.size),'embedding_sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
embedding_manifest = pd.DataFrame(emb_rows)
if embedding_manifest.dimension.nunique() != 1:
    raise ValueError('Embedding dimensions differ across test IDs')
selected_files = files[:SMOKE_PROTEINS] if SMOKE_TEST else files
protein_ids = [p.stem for p in selected_files]
print('Embeddings verified:',len(files),'| vector dimension:',emb_rows[0]['dimension'])
print('Selected for generation:',len(protein_ids),'| first:',protein_ids[:6], '| last:',protein_ids[-3:])
print('Maximum new PyTorch accepted molecules:',len(protein_ids)*len(MODEL_NAMES)*effective_runs*effective_target)
display(embedding_manifest.head(6))

## 4 · Ordered per-model atom vocabulary (legacy vs new 55-type vocabulary)

In [ ]:
MAX_ITER_PER_MOL = 120
ATOM_TYPES_FILE = None
_mol_spec = None

def set_atom_vocab(path):
    global ATOM_TYPES_FILE, _mol_spec
    ATOM_TYPES_FILE = Path(path)
    if not ATOM_TYPES_FILE.is_file():
        raise FileNotFoundError(f'Atom vocabulary is required: {ATOM_TYPES_FILE}')
    _mol_spec = MoleculeSpec(ATOM_TYPES_FILE)
    return _mol_spec

class MoleculeSpec:
    def __init__(self, file_name):
        rows = []
        with Path(file_name).open(encoding='utf8') as f:
            for i,line in enumerate(f,1):
                if not line.strip(): continue
                parts = line.strip().split(',')
                if len(parts)!=3: raise ValueError(f'{file_name}:{i}: expected symbol,charge,explicitHs')
                rows.append((parts[0], int(parts[1]), int(parts[2])))
        if not rows or len(rows)!=len(set(rows)):raise ValueError('Empty or duplicated atom vocabulary')
        self.atom_types=rows
        self.atom_index={t:i for i,t in enumerate(rows)}
        self.bond_orders=[Chem.BondType.AROMATIC,Chem.BondType.SINGLE,Chem.BondType.DOUBLE,Chem.BondType.TRIPLE]
        self.bond_index={b:i for i,b in enumerate(self.bond_orders)}
        self.num_atom_types=len(self.atom_types)
        self.num_bond_types=len(self.bond_orders)
        self.max_iter=MAX_ITER_PER_MOL

    def get_atom_type(self, a):
        key=(a.GetSymbol(), a.GetFormalCharge(), a.GetNumExplicitHs())
        if key not in self.atom_index:raise ValueError(f'Out-of-vocabulary atom {key}; check atom_types.txt')
        return self.atom_index[key]

    def get_bond_type(self, b):
        bt=b.GetBondType()
        if bt not in self.bond_index:raise ValueError(f'Unsupported bond {bt}')
        return self.bond_index[bt]

    def index_to_atom(self, idx):
        symbol,charge,hs=self.atom_types[int(idx)]
        atom=Chem.Atom(symbol);atom.SetFormalCharge(charge);atom.SetNumExplicitHs(hs)
        return atom

    def index_to_bond(self, mol, begin_id, end_id, idx):
        mol.AddBond(int(begin_id),int(end_id),self.bond_orders[int(idx)])


def get_mol_spec():
    global _mol_spec
    if _mol_spec is None:_mol_spec=MoleculeSpec(ATOM_TYPES_FILE)
    return _mol_spec


def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for c in iter(lambda:f.read(1<<20),b''):h.update(c)
    return h.hexdigest()


def get_d(A, X):

    _to_sparse = lambda _A, _X: sparse.coo_matrix((np.ones([_A.shape[0] * 2], dtype=np.int32),
                                                   (np.concatenate([_A[:, 0], _A[:, 1]], axis=0),
                                                    np.concatenate([_A[:, 1], _A[:, 0]], axis=0))),
                                                    shape=[_X.shape[0], ] * 2)   # this is creating a square matrtix of size len(X)


    A_sparse = _to_sparse(A, X)

    d2 = A_sparse * A_sparse
    d3 = d2 * A_sparse

    D_2 = np.stack(d2.nonzero(), axis=1)
    D_2 = D_2[D_2[:, 0] < D_2[:, 1], :]

    D_3 = np.stack(d3.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    D_3_sparse = _to_sparse(D_3, X)
    D_3_sparse = D_3_sparse - D_3_sparse.multiply(A_sparse)
    D_3 = np.stack(D_3_sparse.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    return D_2, D_3


def get_mol_from_graph(X, A, sanitize=True):
    try:
        mol = Chem.RWMol(Chem.Mol())

        X, A = X.tolist(), A.tolist()
        for i, atom_type in enumerate(X):
            mol.AddAtom(get_mol_spec().index_to_atom(atom_type))

        for atom_id1, atom_id2, bond_type in A:
            get_mol_spec().index_to_bond(mol, atom_id1, atom_id2, bond_type)
    except:
        return None

    if sanitize:
        try:
            mol = mol.GetMol()
            Chem.SanitizeMol(mol)
            return mol
        except:
            return None
    else:
        return mol


def get_mol_from_graph_list(graph_list, sanitize=True):
    mol_list = [get_mol_from_graph(X, A, sanitize) for X, A in graph_list]
    return mol_list


In [ ]:
spec = set_atom_vocab(LEGACY_ATOM_TYPES)
print('Atom types:', spec.num_atom_types, '| bond types:', spec.num_bond_types)
print('Atom vocabulary SHA-256:', file_hash(LEGACY_ATOM_TYPES))
display(pd.DataFrame(spec.atom_types, columns=['symbol','charge','explicit_Hs']).head(12))

## 5 · Exact PyTorch architectures: legacy, new clean, and conditional fine-tuned

In [ ]:
def sum_segments(x, seg_id, count):
    result=x.new_zeros((count,)+x.shape[1:])
    return result.index_add(0,seg_id,x)


class LinearBN(nn.Module):
    def __init__(self, fin, fout):
        super().__init__()
        self.linear=nn.Linear(fin,fout,bias=False)
        self.bn=nn.BatchNorm1d(fout)
    def forward(self,x):return self.bn(self.linear(x))


class GraphConv(nn.Module):
    """CDGCN GC with one self term + one matrix per bond/distance relation."""
    def __init__(self, fin, fout, n_relations):
        super().__init__()
        self.W=nn.Parameter(torch.empty(fin*(1+n_relations),fout))
        nn.init.xavier_uniform_(self.W)
        self.n_relations=n_relations

    def forward(self,X,edge_lists):
        assert len(edge_lists)==self.n_relations
        terms=[X]
        for edge in edge_lists:
            if edge.numel()==0:terms.append(torch.zeros_like(X))
            else:
                terms.append(sum_segments(X[edge[1]],edge[0],X.shape[0]))
        return torch.cat(terms,dim=-1) @ self.W


class GraphBackbone(nn.Module):
    def __init__(self,n_atom,n_bond,D,F_e,F_h,F_skip,F_c,dense_relu=False):
        super().__init__()
        self.dense_relu=dense_relu
        self.atom_embedding=nn.Embedding(n_atom,F_e)
        self.mask_embedding=nn.Embedding(3,F_e)
        self.convs=nn.ModuleList()
        self.pre_bns=nn.ModuleList()
        for i,fout in enumerate(F_h):
            fin=F_e if i==0 else F_h[i-1]
            self.pre_bns.append(nn.Identity() if i==0 else nn.BatchNorm1d(fin))
            self.convs.append(GraphConv(fin,fout,n_bond+D))
        self.bn_skip=nn.BatchNorm1d(sum(F_h))
        self.skip=LinearBN(sum(F_h),F_skip)
        self.dense=nn.ModuleList()
        for fin,fout in zip([F_skip]+F_c[:-1],F_c):self.dense.append(LinearBN(fin,fout))

    def forward(self,b):
        X=self.atom_embedding(b['X'])+self.mask_embedding(b['last_mask'])
        layer_out=[]
        for conv,bn in zip(self.convs,self.pre_bns):
            X=conv(F.relu(bn(X)) if not isinstance(bn,nn.Identity) else X,b['edges'])
            layer_out.append(X)
        X=F.relu(self.skip(F.relu(self.bn_skip(torch.cat(layer_out,dim=-1)))))
        for dense in self.dense:
            X=dense(X)
            if self.dense_relu:
                X=F.relu(X)
        return X


class Policy(nn.Module):
    def __init__(self,fin,hidden,n_atom,n_bond):
        super().__init__()
        self.n_atom=n_atom;self.n_bond=n_bond
        self.linear_h=LinearBN(fin*2,hidden)
        self.linear_h_t=LinearBN(fin,hidden)
        self.linear_x=nn.Linear(hidden,n_bond*(n_atom+1))
        self.linear_x_t=nn.Linear(hidden,1)

    def forward(self,X,b,X_end):
        nx_rep=b['NX_rep'];n_step=b['NX'].shape[0]
        node_state=torch.cat([X,X_end[nx_rep]],dim=-1)
        node_logits=self.linear_x(F.relu(self.linear_h(node_state)))
        end_logits=self.linear_x_t(F.relu(self.linear_h_t(X_end)))
        # Stable per-graph softmax over every (node, append/connect action) AND end.
        per_node_max=node_logits.max(dim=-1).values
        group_max=end_logits[:,0].clone()
        group_max=group_max.scatter_reduce(0,nx_rep,per_node_max,reduce='amax',include_self=True)
        node_weight=torch.exp(node_logits-group_max[nx_rep,None])
        end_weight=torch.exp(end_logits-group_max[:,None])
        denom=sum_segments(node_weight.sum(-1,keepdim=True),nx_rep,n_step)+end_weight
        soft_node=node_weight/denom[nx_rep]
        soft_end=(end_weight/denom)[:,0]
        connect=soft_node[:,:self.n_bond]
        append=soft_node[:,self.n_bond:].reshape(-1,self.n_atom,self.n_bond)
        return append,connect,soft_end


class CTDDGPretrainer(nn.Module):
    def __init__(self,n_atom,n_bond,D=2,F_e=16,F_h=(32,64,128,128,256,256),
                 F_skip=256,F_c=(512,),F_policy=128,n_rnn=3,dense_relu=False):
        super().__init__()
        self.n_atom=n_atom;self.n_bond=n_bond
        self.backbone=GraphBackbone(n_atom,n_bond,D,F_e,list(F_h),F_skip,list(F_c),dense_relu=dense_relu)
        self.rnn=nn.GRU(input_size=2*F_c[-1],hidden_size=F_c[-1],num_layers=n_rnn,batch_first=True)
        self.policy=Policy(F_c[-1],F_policy,n_atom,n_bond)
        self.init_atom_logits=nn.Parameter(torch.zeros(n_atom))
        # Original MXNet notebook initializes trainable matrices with Xavier.
        # Match that policy rather than PyTorch's default Kaiming/embedding init.
        for module in self.modules():
            if isinstance(module,(nn.Linear,nn.Embedding)):
                nn.init.xavier_uniform_(module.weight)
                if getattr(module,'bias',None) is not None:nn.init.zeros_(module.bias)
            elif isinstance(module,nn.GRU):
                for name,param in module.named_parameters():
                    if 'weight' in name:nn.init.xavier_uniform_(param)
                    elif 'bias' in name:nn.init.zeros_(param)
            elif isinstance(module,nn.BatchNorm1d):
                nn.init.ones_(module.weight);nn.init.zeros_(module.bias)

    def per_molecule_loglik(self,b):
        B,K=b['batch_size'],b['iw_size']
        X=self.backbone(b)
        n_step=b['NX'].numel()
        X_avg=sum_segments(X,b['NX_rep'],n_step)/b['NX'][:,None].to(X.dtype)
        X_last=X[b['NX_cum']-1]
        features=torch.cat([X_avg,X_last],dim=-1)
        # Padded sequences; gathering only actual step outputs means padding cannot change valid outputs.
        padded=features[b['graph_to_rnn']].reshape(B*K,-1,features.shape[-1])
        rnn_out,_=self.rnn(padded)
        rnn_out=rnn_out.reshape(B,K,-1,rnn_out.shape[-1])
        idx=b['rnn_to_graph']
        X_end=rnn_out[idx[:,0],idx[:,1],idx[:,2]]
        append,connect,end=self.policy(X,b,X_end)
        act=b['actions'];act_type=act[:,0]
        eps=1e-10
        init_logprob=F.log_softmax(self.init_atom_logits,dim=0)
        init_ll=init_logprob[b['action_0']]
        # indices in actions refer to global offset into concatenated graph nodes
        append_log=torch.log(append[act[:,3],act[:,1],act[:,2]]+eps)
        connect_log=torch.log(connect[act[:,4],act[:,2]]+eps)
        end_log=torch.log(end+eps)
        action_ll=torch.where(act_type==0,append_log,torch.where(act_type==1,connect_log,end_log))
        path_ll=sum_segments(action_ll[:,None],b['iw_ids'],B*K)[:,0] + init_ll
        path_ll=path_ll.reshape(B,K)-b['log_p_sigma'].reshape(B,K)
        return torch.logsumexp(path_ll,dim=1)-math.log(K)



print("PyTorch CTDDG architecture definitions loaded: GraphBackbone / Policy / CTDDGPretrainer")

class ConditionalGraphBackbone(GraphBackbone):
    """Matches ConditionalGraphBackbone from the supplied BindingDB fine-tuner."""
    def __init__(self,n_atom,n_bond,D,F_e,F_h,F_skip,F_c,n_c):
        super().__init__(n_atom,n_bond,D,F_e,F_h,F_skip,F_c,dense_relu=False)
        self.n_c=n_c
        self.protein_projections=nn.ModuleList([
            nn.Linear(n_c,width,bias=False) for width in F_h
        ])
        for layer in self.protein_projections:
            nn.init.xavier_uniform_(layer.weight)

    def forward(self,b):
        c=b['c']
        molecule_ids=b['mol_ids_rep']
        assert c.ndim==2 and c.shape[1]==self.n_c
        assert molecule_ids.numel()==b['X'].numel()
        X=self.atom_embedding(b['X'])+self.mask_embedding(b['last_mask'])
        layer_out=[]
        protein_features=[projection(c) for projection in self.protein_projections]
        for conv,bn,pfeat in zip(self.convs,self.pre_bns,protein_features):
            xin=F.relu(bn(X)) if not isinstance(bn,nn.Identity) else X
            X=conv(xin,b['edges'])+pfeat[molecule_ids]
            layer_out.append(X)
        X=F.relu(self.skip(F.relu(self.bn_skip(torch.cat(layer_out,dim=-1)))))
        for dense in self.dense:
            X=dense(X)  # Exactly as in the fine-tuning notebook: no final ReLU
        return X


class ConditionalCTDDG(CTDDGPretrainer):
    """Protein-conditioned model; parameter names match fine-tuning checkpoints."""
    def __init__(self,model_config,n_c=6165):
        super().__init__(**model_config)
        self.backbone=ConditionalGraphBackbone(
            model_config['n_atom'],model_config['n_bond'],model_config['D'],
            model_config['F_e'],list(model_config['F_h']),model_config['F_skip'],
            list(model_config['F_c']),n_c
        )
        n_atom=model_config['n_atom']
        head_width=3*n_atom
        self.init_head=nn.Sequential(
            nn.Linear(n_c,head_width,bias=False),
            nn.BatchNorm1d(head_width),
            nn.ReLU(),
            nn.Linear(head_width,n_atom),
        )
        for layer in self.init_head.modules():
            if isinstance(layer,nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
        self.conditional=True



In [ ]:
@dataclass
class _MolState:
    atoms: list
    bonds: list
    hidden: torch.Tensor
    last_action: int = 1          # 1 = init/append, 2 = connect
    actions: int = 0
    done: bool = False
    status: str = 'running'
    protein_vector: torch.Tensor | None = None


def _graph_batch(states, device, bond_count=4):
    """Build EXACT same bond + D2 + D3 relation arrays as training's get_d()."""
    NX = np.asarray([len(st.atoms) for st in states], dtype=np.int64)
    X = np.asarray([atom for st in states for atom in st.atoms], dtype=np.int64)
    offsets = np.cumsum(np.pad(NX, (1, 0)))
    bond_rows = []
    mask = np.zeros((len(X),), dtype=np.int64)
    for i, st in enumerate(states):
        mask[offsets[i+1] - 1] = st.last_action
        bond_rows.extend((offsets[i]+u, offsets[i]+v, b) for u, v, b in st.bonds)
    A = np.asarray(bond_rows, dtype=np.int64).reshape((-1, 3))
    # Preserve the source pretraining exact sparse-walk definition; these are
    # NOT necessarily shortest-distance=2/3 sets.
    d2, d3 = get_d(A, X)
    relation_pairs = [A[A[:, 2] == b, :2] for b in range(bond_count)] + [d2, d3]
    edge_tensors = []
    for pairs in relation_pairs:
        if not len(pairs):
            edge_tensors.append(torch.empty((2, 0), dtype=torch.long, device=device))
        else:
            both = np.concatenate([pairs, pairs[:, ::-1]], axis=0)
            edge_tensors.append(torch.tensor(both.T.copy(), dtype=torch.long, device=device))
    return {'X': torch.tensor(X, dtype=torch.long, device=device),
            'last_mask': torch.tensor(mask, dtype=torch.long, device=device),
            'mask': torch.tensor(mask, dtype=torch.long, device=device),
            'edges': edge_tensors,
            'NX': torch.tensor(NX, dtype=torch.long, device=device),
            'NX_rep': torch.repeat_interleave(torch.arange(len(NX), device=device),
                                               torch.tensor(NX, dtype=torch.long, device=device)),
            'NX_cum': torch.tensor(offsets[1:], dtype=torch.long, device=device),
            **({'c': torch.stack([st.protein_vector for st in states],dim=0),
                'mol_ids_rep': torch.repeat_interleave(
                    torch.arange(len(NX),device=device),
                    torch.tensor(NX,dtype=torch.long,device=device))}
               if states[0].protein_vector is not None else {})}


@torch.inference_mode()
def _decode_probabilities(model, states):
    """Single incremental GRU step for each unfinished molecular graph.

    Mirrors the model's *training* feature order [mean node embedding, last node
    embedding], the original MXNet stepwise inference, and the trained Policy.
    """
    if not states:
        return []
    device = next(model.parameters()).device
    batch = _graph_batch(states, device, model.n_bond)
    X = model.backbone(batch)
    n = batch['NX'].numel()
    mean_X = sum_segments(X, batch['NX_rep'], n) / batch['NX'][:, None].to(X.dtype)
    last_X = X[batch['NX_cum'] - 1]
    features = torch.cat([mean_X, last_X], dim=-1).unsqueeze(1)
    h0 = torch.stack([st.hidden for st in states], dim=1).contiguous()
    next_X, h1 = model.rnn(features, h0)
    ap, cn, en = model.policy(X, batch, next_X[:, 0, :])
    offsets = torch.cat([batch['NX'].new_zeros(1), batch['NX_cum']]).tolist()
    results = []
    for i, st in enumerate(states):
        st.hidden = h1[:, i, :].clone()
        sl = slice(offsets[i], offsets[i+1])
        results.append((ap[sl], cn[sl], en[i]))
    return results


def _draw_action(append, connect, end, generator=None, greedy=False):
    """Sample all (node, atom, bond), (node,bond), and END using joint probabilities."""
    n_app = append.numel()
    n_con = connect.numel()
    flat = torch.cat([append.reshape(-1), connect.reshape(-1), end.reshape(1)])
    if not torch.isfinite(flat).all() or torch.any(flat < 0):
        raise FloatingPointError('Invalid action distribution (NaN/Inf/negative probabilities)')
    z = flat.sum()
    if not torch.isfinite(z) or z.item() <= 0:
        raise FloatingPointError('Zero/invalid action probability mass')
    # Policy already normalizes all actions, but correct round-off prior to sampling.
    p = flat / z
    index = int(torch.argmax(p).item()) if greedy else int(torch.multinomial(p, 1, generator=generator).item())
    if index == n_app + n_con:
        return ('end',)
    if index >= n_app:
        local = index - n_app
        return ('connect', local // connect.shape[-1], local % connect.shape[-1])
    bond_count = append.shape[-1]
    atom_count = append.shape[-2]
    return ('append', index // (atom_count*bond_count),
            (index // bond_count) % atom_count, index % bond_count)



print("Graph builder, stepwise GRU decoder, and categorical action sampler defined.")

## 6 · Strict model checkpoint loading and training-run provenance verification

In [ ]:
DEFAULT_CONFIG={'n_atom':0,'n_bond':4,'D':2,'F_e':16,
                'F_h':[32,64,128,128,256,256],'F_skip':256,
                'F_c':[512],'F_policy':128,'n_rnn':3}
CONFIG_KEYS=tuple(DEFAULT_CONFIG)


def read_checkpoint(model_name):
    """Load model, model-specific vocab, and architecture with strict state check."""
    source=MODEL_SOURCES[model_name]
    vocab=Path(source['vocab'])
    checkpoint_path=next((p for p in source['candidates'] if p.is_file()),None)
    if checkpoint_path is None:
        raise FileNotFoundError(f'No checkpoint for {model_name}; searched {source["candidates"]}')
    print('\nLoading',model_name,':',checkpoint_path)
    spec=set_atom_vocab(vocab)
    vocab_hash=file_hash(vocab)

    try:
        obj=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    except Exception as exc:
        if not TRUST_LOCAL_CHECKPOINT:
            raise RuntimeError('Explicit trusted checkpoint flag required for this file') from exc
        print('Trusted local checkpoint requires weights_only=False:',type(exc).__name__)
        obj=torch.load(checkpoint_path,map_location='cpu',weights_only=False)

    if not isinstance(obj,dict):
        raise TypeError('Expected checkpoint dictionary')
    state=next((obj[k] for k in ('model','model_state_dict','state_dict')
                if isinstance(obj.get(k),dict)),obj)
    if not state or not all(isinstance(k,str) and torch.is_tensor(v) for k,v in state.items()):
        raise TypeError('Checkpoint does not contain a valid PyTorch state_dict')
    if all(k.startswith('module.') for k in state):
        state={k[7:]:v for k,v in state.items()}

    kind=source['kind']
    metadata={}
    cfg=DEFAULT_CONFIG.copy()
    cfg['n_atom']=spec.num_atom_types
    cfg['n_bond']=spec.num_bond_types

    if kind=='clean_unconditional':
        run_config_file=source['metadata_dir']/'run_config.json'
        if not run_config_file.is_file():
            run_config_file=MODEL_ROOT/model_name/'run_config.json'
        assert run_config_file.is_file(), f'Missing training run_config.json: {run_config_file}'
        run_config=json.loads(run_config_file.read_text())
        locked=run_config['locked_config']
        cfg.update({k:locked['model_config'][k] for k in CONFIG_KEYS if k in locked['model_config']})
        assert locked['vocabulary_sha256']==vocab_hash, 'New CDGCN vocab hash differs from training'
        assert obj.get('signature')==run_config['signature'], 'New checkpoint signature mismatch'
        assert int(obj.get('completed_steps',0))>0, 'New pretraining checkpoint has no completed steps'
        metadata={'completed_steps':obj['completed_steps'],'run_signature':run_config['signature']}
    elif kind=='bindingdb_conditional':
        meta_file=source['metadata_dir']/'run_meta.json'
        assert meta_file.is_file(), f'Missing fine-tuning run_meta.json: {meta_file}'
        metadata=json.loads(meta_file.read_text())
        signature=metadata['signature']
        assert obj.get('signature')==signature, 'Fine-tuning checkpoint signature differs from run metadata'
        assert signature['atom_types_sha256']==vocab_hash, 'Fine-tuned vocab hash differs from training'
        assert signature['pretraining']=='cdgcn_nll', 'Unexpected fine-tuning source pretraining'
        assert signature['finetuning_objective']=='ctddg_eq8', 'Unexpected fine-tuning objective'
        assert signature['protein_size']==source['n_c'], 'Fine-tuned embedding dimension mismatch'
        cfg.update({k:signature['model_config'][k] for k in CONFIG_KEYS
                    if k in signature['model_config']})
        metadata={'signature':signature,
                  'step':obj.get('step',obj.get('state',{}).get('step')),
                  'checkpoint':checkpoint_path.name}
    else:
        directory=source['metadata_dir']
        for filename in ('run_meta.json','model_config.json'):
            p=directory/filename
            if p.is_file():
                data=json.loads(p.read_text())
                if isinstance(data,dict):metadata.update(data)
        for expected_name in ('atom_types_sha256','atom_vocab_sha256'):
            expected=metadata.get(expected_name)
            if expected and expected!=vocab_hash:
                raise ValueError(f'{model_name} ordered vocabulary hash mismatch: {expected_name}')
        saved=next((m for m in (metadata.get('model_config'),metadata.get('config'),metadata)
                    if isinstance(m,dict) and 'n_atom' in m),None)
        if saved:cfg.update({k:saved[k] for k in CONFIG_KEYS if k in saved})
        else:print('No archived legacy model_config; testing reference widths via strict state load')

    assert cfg['n_atom']==spec.num_atom_types, f'{model_name}: wrong vocabulary size'
    assert cfg['n_bond']==spec.num_bond_types==4
    assert cfg['D']==2, 'Generator only implements D=2'
    if kind=='bindingdb_conditional':
        model=ConditionalCTDDG(cfg,n_c=source['n_c'])
    else:
        model=CTDDGPretrainer(**cfg,dense_relu=source['dense_relu'])
        model.conditional=False
    model.load_state_dict(state,strict=True)  # never ignore missing/unexpected weights
    for name,tensor in model.state_dict().items():
        if torch.is_floating_point(tensor) and not torch.isfinite(tensor).all():
            raise FloatingPointError(f'NaN/Inf checkpoint tensor: {name}')
    model=model.to(DEVICE).eval()
    print('PASS strict load | protein-conditioned:',model.conditional,
          '| atom types:',spec.num_atom_types,'| weight SHA256:',file_hash(checkpoint_path)[:16])
    return model,checkpoint_path,metadata,cfg


for model_name in MODEL_NAMES:
    src=MODEL_SOURCES[model_name]
    print(model_name, '| checkpoint found:',
          next((str(p) for p in src['candidates'] if p.is_file()),'NOT FOUND'),
          '| vocab:',src['vocab'])
print('Strict, model-specific checkpoint loader ready.')

## 7 · Graph-to-SMILES export (not an evaluation)

Nitesh's generator saves **unique RDKit-sanitizable molecules**. This conversion is required to reproduce that CSV format. The full unfiltered attempts are separately preserved for subsequent chemical-validity analysis.

In [ ]:
def convert_graph_for_output(st):
    """Attempt RDKit graph->canonical SMILES conversion; no metric is calculated."""
    try:
        rw = Chem.RWMol()
        for atom_id in st.atoms:
            rw.AddAtom(spec.index_to_atom(int(atom_id)))
        for u, v, b in st.bonds:
            rw.AddBond(int(u), int(v), spec.bond_orders[int(b)])
        mol = rw.GetMol()
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True), None
    except Exception as exc:
        return None, f'{type(exc).__name__}: {str(exc)[:180]}'

print('Graph-to-SMILES export conversion ready (no evaluation metrics).')

## 8 · Sample complete graphs: unconditional or protein-conditioned decoding

In [ ]:
@torch.inference_mode()
def sample_batch(model, how_many, generator, protein_vector=None):
    if getattr(model,'conditional',False):
        assert protein_vector is not None, 'Conditional generation requires a protein vector'
        assert protein_vector.ndim==1 and protein_vector.numel()==model.backbone.n_c
        # The fine-tuner uses this protein-dependent first-atom distribution.
        p0=F.softmax(model.init_head(protein_vector.unsqueeze(0))[0],dim=0)
    else:
        p0=F.softmax(model.init_atom_logits,dim=0)
    if not torch.isfinite(p0).all() or not (p0 >= 0).all():
        raise FloatingPointError('Invalid first-atom probability distribution')
    if GREEDY:
        choices = [int(torch.argmax(p0).item())] * how_many
    else:
        choices = torch.multinomial(p0, how_many, replacement=True, generator=generator).tolist()
    states = [_MolState(atoms=[int(a)], bonds=[],
                        hidden=torch.zeros(model.rnn.num_layers, model.rnn.hidden_size, device=DEVICE),
                        protein_vector=protein_vector)
              for a in choices]
    for step in range(MAX_ACTIONS):
        active = [s for s in states if not s.done]
        if not active:
            break
        distributions = _decode_probabilities(model, active)
        for st, (append, connect, end) in zip(active, distributions):
            try:
                action = _draw_action(append, connect, end, generator=generator, greedy=GREEDY)
            except (RuntimeError, ValueError, FloatingPointError) as ex:
                st.done = True
                st.status = 'invalid_action_distribution'
                continue
            st.actions += 1
            if action[0] == 'end':
                st.done = True
                st.status = 'ended'
            elif action[0] == 'append':
                _, target, atom, bond = action
                if len(st.atoms) >= MAX_ATOMS:
                    st.done = True
                    st.status = 'max_atoms_exceeded'
                else:
                    st.bonds.append((int(target), len(st.atoms), int(bond)))
                    st.atoms.append(int(atom))
                    st.last_action = 1
            elif action[0] == 'connect':
                _, target, bond = action
                target, bond = int(target), int(bond)
                last = len(st.atoms) - 1
                if target == last:
                    st.done = True
                    st.status = 'self_bond'
                elif any({u, v} == {last, target} for u, v, _ in st.bonds):
                    st.done = True
                    st.status = 'duplicate_bond'
                else:
                    st.bonds.append((last, target, bond))
                    st.last_action = 2
    for st in states:
        if not st.done:
            st.done, st.status = True, 'max_actions_exceeded'
    return states


def stable_seed(*parts):
    key = '|'.join(map(str, parts))
    return int(hashlib.sha256(key.encode()).hexdigest()[:8], 16) % (2**31-1)


def atomic_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def collect_unique_samples(model, model_name, protein_id, run_id, target, max_attempts,
                           protein_vector=None, raw_path=None, accepted_path=None):
    seed = stable_seed(SEED, model_name, protein_id, run_id)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed(seed)
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(seed)

    chosen, seen, attempts = [], set(), []
    t0 = time.monotonic()
    batch_num = 0
    while len(chosen) < target and len(attempts) < max_attempts:
        batch_num += 1
        count = min(BATCH_SIZE, max_attempts - len(attempts))
        states = sample_batch(model, count, generator, protein_vector=protein_vector)
        for st in states:
            smiles, error = convert_graph_for_output(st)
            accepted = (st.status == 'ended' and smiles is not None and
                        smiles not in seen and len(chosen) < target)
            if accepted:
                seen.add(smiles)
                chosen.append({'Run_ID': run_id, 'SMILES': smiles})
            attempts.append({
                'Run_ID': run_id, 'Attempt_ID': len(attempts) + 1,
                'SMILES': smiles, 'decode_status': st.status,
                'saved_in_main_csv': bool(accepted),
                'atom_indices_json': json.dumps(st.atoms),
                'bond_indices_json': json.dumps(st.bonds),
                'conversion_error': error,
            })
        # Keep intermediates available if generation is interrupted.
        if batch_num % SAVE_RAW_EVERY_BATCHES == 0 or len(chosen) >= target:
            if raw_path is not None:
                atomic_csv(pd.DataFrame(attempts), raw_path)
            if accepted_path is not None:
                atomic_csv(pd.DataFrame(chosen, columns=['Run_ID', 'SMILES']), accepted_path)
        if (batch_num == 1 or batch_num % SHOW_PROGRESS_EVERY_BATCHES == 0 or
            len(chosen) >= target or len(attempts) >= max_attempts):
            print(f'    Run {run_id}: {len(chosen):>3}/{target} samples collected | '
                  f'{len(attempts):>4} attempts | '
                  f'{time.monotonic()-t0:.1f}s', flush=True)

    if raw_path is not None:
        atomic_csv(pd.DataFrame(attempts), raw_path)
    if accepted_path is not None:
        atomic_csv(pd.DataFrame(chosen, columns=['Run_ID','SMILES']), accepted_path)
    return pd.DataFrame(chosen, columns=['Run_ID', 'SMILES']), pd.DataFrame(attempts)


## 9 · Produce crash-safe, resumable, non-overwriting 10-run outputs for four PyTorch models

In [ ]:
def _json_write_atomic(payload, path):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(payload,indent=2,default=str))
    tmp.replace(path)


def _check_run_snapshot(run_dir, *, model_name, protein_id, run_id, expected_seed,
                        weight_hash, vocab_hash, target, max_attempts,
                        conditioned=False, embedding_hash=None):
    accepted_file=run_dir/'accepted.csv'
    raw_file=run_dir/'raw_attempts.csv'
    info_file=run_dir/'run_info.json'
    paths=[accepted_file,raw_file,info_file]
    if not any(p.exists() for p in paths):
        return None
    if not all(p.is_file() for p in paths):
        raise FileExistsError(f'Incomplete/uncertified run snapshot: {run_dir}')
    meta=json.loads(info_file.read_text())
    required={'model':model_name,'protein_id':protein_id,'Run_ID':run_id,
              'seed':expected_seed,'checkpoint_sha256':weight_hash,
              'atom_types_sha256':vocab_hash,'target':target,
              'max_attempts':max_attempts,'max_actions':MAX_ACTIONS,'max_atoms':MAX_ATOMS,
              'greedy':GREEDY,'batch_size':BATCH_SIZE}
    if conditioned:
        required.update(conditioned_on_protein=True,protein_embedding_sha256=embedding_hash)
    if any(meta.get(k)!=v for k,v in required.items()):
        raise ValueError(f'Run snapshot identity/configuration mismatch: {run_dir}')
    if meta.get('accepted_sha256')!=file_hash(accepted_file) or meta.get('raw_sha256')!=file_hash(raw_file):
        raise ValueError(f'Run snapshot checksum mismatch: {run_dir}')
    accepted=pd.read_csv(accepted_file,keep_default_na=False)
    raw=pd.read_csv(raw_file,keep_default_na=False)
    if list(accepted.columns)!=['Run_ID','SMILES'] or len(accepted)!=target or accepted.SMILES.nunique()!=target:
        raise ValueError(f'Accepted molecules in snapshot invalid: {run_dir}')
    fields={'Run_ID','Attempt_ID','SMILES','decode_status','saved_in_main_csv'}
    if not fields.issubset(raw.columns) or len(raw)==0:
        raise ValueError(f'Raw-attempt records invalid: {run_dir}')
    if not (raw.Run_ID==run_id).all() or not (accepted.Run_ID==run_id).all():
        raise ValueError(f'Wrong run ID: {run_dir}')
    if len(raw)!=meta.get('attempts') or len(accepted)!=meta.get('accepted'):
        raise ValueError(f'Incorrect saved run counts: {run_dir}')
    if raw.Attempt_ID.tolist()!=list(range(1,len(raw)+1)):
        raise ValueError(f'Incorrect attempt IDs: {run_dir}')
    from collections import Counter
    flags=raw.saved_in_main_csv.astype(str).str.lower().isin(('true','1'))
    if Counter(raw.loc[flags,'SMILES'])!=Counter(accepted.SMILES):
        raise ValueError(f'Accepted/sample log mismatch: {run_dir}')
    return accepted,raw,meta


def _check_completed_protein(dest, protein_id, *, model_name,
                             weight_hash, vocab_hash, embedding_hash,
                             conditioned):
    """Only skip a completed dataset with verified counts, hashes, provenance."""
    main=dest/f'{protein_id}_all_samples.csv'
    raw=dest/f'{protein_id}_raw_attempts.csv'
    info=dest/f'{protein_id}_generation_info.json'
    if not (main.is_file() and raw.is_file() and info.is_file()):
        return False
    meta=json.loads(info.read_text())
    if meta.get('checkpoint_sha256')!=weight_hash or meta.get('atom_types_sha256')!=vocab_hash:
        raise ValueError(f'Existing samples use a different checkpoint or vocabulary: {dest}')
    if conditioned:
        if (meta.get('conditioned_on_protein') is not True
                or meta.get('protein_embedding_sha256')!=embedding_hash):
            raise ValueError(f'Existing conditional sample embedding provenance mismatch: {dest}')
    elif meta.get('conditioned_on_protein',False):
        raise ValueError(f'Unexpected conditional samples in unconditional folder: {dest}')
    if meta.get('total_runs')!=effective_runs or meta.get('target_per_run')!=effective_target:
        return False
    a=pd.read_csv(main,keep_default_na=False)
    r=pd.read_csv(raw,keep_default_na=False)
    if not {'Run_ID','SMILES'}.issubset(a.columns):
        raise ValueError(f'Bad accepted CSV format: {main}')
    if not {'Run_ID','Attempt_ID','SMILES','decode_status','saved_in_main_csv'}.issubset(r.columns):
        raise ValueError(f'Bad raw CSV format: {raw}')
    if len(a)!=effective_runs*effective_target or len(r)==0:
        return False
    if meta.get('samples_saved')!=len(a) or meta.get('total_attempts')!=len(r):
        return False
    if a.groupby('Run_ID').size().to_dict()!={i:effective_target for i in range(1,effective_runs+1)}:
        return False
    if any(group.SMILES.nunique()!=effective_target for _,group in a.groupby('Run_ID')):
        return False
    if set(r.Run_ID)!=set(range(1,effective_runs+1)):
        return False
    from collections import Counter
    for rid in range(1,effective_runs+1):
        ar=a[a.Run_ID==rid]
        rr=r[r.Run_ID==rid]
        flags=rr.saved_in_main_csv.astype(str).str.lower().isin(('true','1'))
        if Counter(rr.loc[flags,'SMILES'])!=Counter(ar.SMILES):
            raise ValueError(f'Existing accepted/raw sample mismatch: {dest}, run {rid}')
    return True


def _audit_mxnet_existing():
    """PyTorch cannot load the MXNet generator. Inspect existing files only."""
    model='nitesh_model'
    present,complete=0,0
    for protein_id in protein_ids:
        f=SAVE_ROOT/model/protein_id/f'{protein_id}_all_samples.csv'
        if not f.is_file():continue
        present+=1
        a=pd.read_csv(f,keep_default_na=False)
        if {'Run_ID','SMILES'}.issubset(a.columns):
            sizes=a.groupby('Run_ID').size().to_dict()
            if sizes=={i:effective_target for i in range(1,effective_runs+1)}:
                complete+=1
    print(f'MXNet nitesh_model: {complete}/{len(protein_ids)} complete sample CSVs '
          f'({present} present). Not generated or modified by this PyTorch notebook.')
    if complete<len(protein_ids):
        print('NOTE: Run the separate MXNet generator for missing nitesh_model samples before a strict five-model evaluation.')


# Audit first; no model loaded if all outputs for a model are already finished.
SAVE_ROOT.mkdir(parents=True,exist_ok=True)
_audit_mxnet_existing()
all_summaries=[]
for model_name in MODEL_NAMES:
    source=MODEL_SOURCES[model_name]
    checkpoint_path=next((p for p in source['candidates'] if p.is_file()),None)
    if checkpoint_path is None:
        # A model may have been archived after its results were generated.
        # Allow a READ-ONLY skip if all sample CSVs/metadata are already verified.
        # Without the checkpoint itself, its identity cannot be independently rehashed.
        vocab_hash=file_hash(source['vocab'])
        archived=[]
        for protein_id in protein_ids:
            dest=SAVE_ROOT/model_name/protein_id
            meta_file=dest/f'{protein_id}_generation_info.json'
            if not meta_file.is_file():
                break
            meta=json.loads(meta_file.read_text())
            recorded_hash=meta.get('checkpoint_sha256')
            if not isinstance(recorded_hash,str) or len(recorded_hash)!=64:
                break
            emb_hash=embedding_manifest.set_index('protein_id').loc[protein_id,'embedding_sha256']
            if not _check_completed_protein(dest,protein_id,model_name=model_name,
                    weight_hash=recorded_hash,vocab_hash=vocab_hash,
                    embedding_hash=emb_hash,
                    conditioned=source['kind']=='bindingdb_conditional'):
                break
            archived.append(recorded_hash)
        if len(archived)==len(protein_ids) and len(set(archived))==1 and SKIP_COMPLETE:
            print(f'SKIP ENTIRE MODEL {model_name}: complete archived results found. '
                  'WARNING: checkpoint no longer available to rehash.')
            continue
        raise FileNotFoundError(f'{model_name}: checkpoint missing and full output could not be verified: '
                                f'{source["candidates"]}')
    vocab_path=Path(source['vocab'])
    weight_hash=file_hash(checkpoint_path)
    vocab_hash=file_hash(vocab_path)
    conditioned=source['kind']=='bindingdb_conditional'

    # Validate completed proteins BEFORE loading a potentially expensive model.
    pending=[]
    for protein_id in protein_ids:
        dest=SAVE_ROOT/model_name/protein_id
        emb_hash=embedding_manifest.set_index('protein_id').loc[protein_id,'embedding_sha256']
        done=_check_completed_protein(dest,protein_id,model_name=model_name,
            weight_hash=weight_hash,vocab_hash=vocab_hash,
            embedding_hash=emb_hash,conditioned=conditioned)
        if done:
            if not SKIP_COMPLETE:
                raise FileExistsError(f'Completed output exists and SKIP_COMPLETE=False: {dest}')
            print(f'SKIP {model_name}/{protein_id}: verified 10-run output',flush=True)
        else:
            pending.append(protein_id)
    if not pending:
        print(f'SKIP ENTIRE MODEL {model_name}: all {len(protein_ids)} protein slots completed')
        continue

    print('\n'+'='*72+'\nMODEL: '+model_name+' | pending:',len(pending),flush=True)
    model,weight_path,checkpoint_meta,config=read_checkpoint(model_name)
    with torch.inference_mode():
        vec=None
        if conditioned:
            vec=torch.as_tensor(protein_vectors[pending[0]],device=DEVICE)
            assert vec.numel()==source['n_c'], (
                f'{pending[0]}: embedding has {vec.numel()} dims, fine-tuned model requires {source["n_c"]}')
        p0=F.softmax(model.init_head(vec.unsqueeze(0))[0],dim=0) if conditioned else F.softmax(model.init_atom_logits,dim=0)
        state=_MolState([int(torch.argmax(p0).item())],[],
                        torch.zeros(model.rnn.num_layers,model.rnn.hidden_size,device=DEVICE),
                        protein_vector=vec)
        ap,co,en=_decode_probabilities(model,[state])[0]
        total_prob=float((ap.sum()+co.sum()+en).item())
        if not math.isfinite(total_prob) or abs(total_prob-1.0)>1e-4:
            raise ValueError(f'Invalid decoder probability sum: {total_prob}')
    print('PASS: model-specific strict weights and probability normalization',flush=True)
    for i,protein_id in enumerate(pending,1):
        dest=SAVE_ROOT/model_name/protein_id
        runs_dir=dest/'runs'
        runs_dir.mkdir(parents=True,exist_ok=True)
        main_file=dest/f'{protein_id}_all_samples.csv'
        raw_file=dest/f'{protein_id}_raw_attempts.csv'
        final_info=dest/f'{protein_id}_generation_info.json'
        emb_hash=embedding_manifest.set_index('protein_id').loc[protein_id,'embedding_sha256']
        protein_vector=None
        if conditioned:
            protein_vector=torch.as_tensor(protein_vectors[protein_id],device=DEVICE,dtype=torch.float32)
            assert protein_vector.numel()==source['n_c'], (
                f'Wrong ProSE embedding size for {protein_id}: {protein_vector.numel()}')
        print(f'[{i}/{len(pending)}] {model_name}: {protein_id}',flush=True)
        accepted_parts=[];raw_parts=[];run_info=[]
        started=time.monotonic()
        for run_id in range(1,effective_runs+1):
            seed=stable_seed(SEED,model_name,protein_id,run_id)
            snapshot=runs_dir/f'run_{run_id:02d}'
            cached=_check_run_snapshot(snapshot,model_name=model_name,protein_id=protein_id,
                run_id=run_id,expected_seed=seed,weight_hash=weight_hash,
                vocab_hash=vocab_hash,target=effective_target,
                max_attempts=effective_max_attempts,
                conditioned=conditioned,embedding_hash=emb_hash)
            if cached is None:
                if snapshot.exists() and any(snapshot.iterdir()):
                    raise FileExistsError(f'Unverified/partial snapshot exists: {snapshot}')
                # Do not overwrite untracked existing output data with no verified snapshots.
                if not accepted_parts and (main_file.exists() or raw_file.exists() or final_info.exists()):
                    if not any((runs_dir/f'run_{r:02d}'/'run_info.json').is_file()
                               for r in range(1,effective_runs+1)):
                        raise FileExistsError(f'Untracked existing output: {dest}')
                accepted,raw=collect_unique_samples(model,model_name,protein_id,run_id,
                    target=effective_target,max_attempts=effective_max_attempts,
                    protein_vector=protein_vector)
                if len(accepted)!=effective_target:
                    snapshot.mkdir(parents=True,exist_ok=True)
                    atomic_csv(accepted,snapshot/'PARTIAL_accepted.csv')
                    atomic_csv(raw,snapshot/'PARTIAL_raw_attempts.csv')
                    raise RuntimeError(f'{model_name}/{protein_id}/run{run_id}: '
                        f'{len(accepted)}/{effective_target} completed molecules out of {len(raw)} attempts. '
                        'PARTIAL logs preserved; do not report incomplete outputs.')
                snapshot.mkdir(parents=True,exist_ok=True)
                if any(snapshot.iterdir()):
                    raise FileExistsError(f'Existing snapshot data: {snapshot}')
                atomic_csv(accepted,snapshot/'accepted.csv')
                atomic_csv(raw,snapshot/'raw_attempts.csv')
                info={'model':model_name,'protein_id':protein_id,'Run_ID':run_id,
                    'seed':seed,'checkpoint_sha256':weight_hash,'atom_types_sha256':vocab_hash,
                    'target':effective_target,'max_attempts':effective_max_attempts,
                    'max_actions':MAX_ACTIONS,'max_atoms':MAX_ATOMS,'greedy':GREEDY,
                    'batch_size':BATCH_SIZE,'accepted':len(accepted),'attempts':len(raw),
                    'accepted_sha256':file_hash(snapshot/'accepted.csv'),
                    'raw_sha256':file_hash(snapshot/'raw_attempts.csv')}
                if conditioned:
                    info.update(conditioned_on_protein=True,protein_embedding_sha256=emb_hash)
                _json_write_atomic(info,snapshot/'run_info.json')
                print(f'  Run {run_id}/{effective_runs}: NEW '
                      f'{len(accepted)} accepted / {len(raw)} attempts | seed={seed}',flush=True)
            else:
                accepted,raw,info=cached
                print(f'  Run {run_id}/{effective_runs}: VERIFIED PREVIOUS SNAPSHOT',flush=True)
            accepted_parts.append(accepted)
            raw_parts.append(raw)
            run_info.append(info)
            # A committed run is immutable. Combined files may be rebuilt from snapshots.
            atomic_csv(pd.concat(accepted_parts,ignore_index=True),main_file)
            atomic_csv(pd.concat(raw_parts,ignore_index=True),raw_file)
        combined=pd.concat(accepted_parts,ignore_index=True)
        raw_combined=pd.concat(raw_parts,ignore_index=True)
        info={'model':model_name,'framework':'pytorch','protein_id':protein_id,
             'conditioned_on_protein':conditioned,
             'embeddings_used_for_conditioning':conditioned,
             'embedding_file':str(EMBEDDINGS_FOLDER/f'{protein_id}.txt'),
             'embedding_sha256':emb_hash,'protein_embedding_sha256':emb_hash,
             'total_runs':effective_runs,'target_per_run':effective_target,
             'samples_saved':len(combined),'total_attempts':len(raw_combined),
             'elapsed_seconds_this_session':round(time.monotonic()-started,2),
             'per_run':[{'Run_ID':x['Run_ID'],'seed':x['seed'],'samples_saved':x['accepted'],
                         'generation_attempts':x['attempts']} for x in run_info],
             'checkpoint':str(weight_path),'checkpoint_sha256':weight_hash,
             'atom_types_sha256':vocab_hash,'base_seed':SEED,
             'max_actions':MAX_ACTIONS,'max_atoms':MAX_ATOMS,
             'smoke_test':SMOKE_TEST,'samples_csv':str(main_file),
             'raw_attempts_csv':str(raw_file)}
        if conditioned:
            info['fine_tuning_provenance']=checkpoint_meta.get('signature',{}).get('class')
        _json_write_atomic(info,final_info)
        all_summaries.append(info)
        print(f'  -> COMPLETE: {len(combined)} accepted / {len(raw_combined)} attempts',flush=True)
    del model
    if torch.cuda.is_available():torch.cuda.empty_cache()
    gc.collect()

# This inventory summarizes only proteins NEWLY generated in the current session.
if all_summaries:
    summary_df=pd.DataFrame([{k:v for k,v in d.items() if k!='per_run'} for d in all_summaries])
    atomic_csv(summary_df,SAVE_ROOT/'pytorch_generation_inventory_latest_session.csv')
    display(summary_df[['model','protein_id','samples_saved','total_attempts']].head(10))
else:
    print('All pre-existing PyTorch outputs verified and skipped. Nothing was regenerated.')
print('Outputs:',SAVE_ROOT)


## 10 · Inspect saved CSV structure (no validity/toxicity calculation)

This checks only filenames and column layout and previews some molecules. Do your actual chemical validity, toxicity and Table 1/3 analysis in the separate evaluation notebook.

In [ ]:
from rdkit.Chem import Draw
for model_name in MODEL_NAMES:
    protein_id=protein_ids[0]
    main=SAVE_ROOT/model_name/protein_id/f'{protein_id}_all_samples.csv'
    if not main.is_file():
        print('Not generated:',model_name,protein_id)
        continue
    raw=SAVE_ROOT/model_name/protein_id/f'{protein_id}_raw_attempts.csv'
    df=pd.read_csv(main)
    assert list(df.columns)==['Run_ID','SMILES'], (main,list(df.columns))
    print('\n',model_name,'|',main)
    print('Saved:',len(df),'| Raw attempt log:',raw.name, '| exists:',raw.is_file())
    display(df.head(8))
    mols=[Chem.MolFromSmiles(s) for s in df.SMILES.head(8)]
    mols=[m for m in mols if m is not None]
    if mols:
        display(Draw.MolsToGridImage(mols,molsPerRow=4,subImgSize=(170,140)))

## 11 · Output layout and comparison limitations

Generated CSVs are saved to `REWORKED_CODE/testing/Validity_Analysis/Generated_104_10runs/<model>/proteinN/`, matching the evaluation notebook. `proteinN_all_samples.csv` retains accepted molecules; `proteinN_raw_attempts.csv` preserves every attempted graph and `proteinN_generation_info.json` records checkpoint hashes, sample counts and whether embeddings were used for conditioning.

**The 104 files for the unconditional models are independent repeated sampling slots, not 104 protein-conditioned trials.** The BindingDB fine-tuned generator is protein-conditioned through its six GCN projections and the protein-dependent first-atom head. The MXNet `nitesh_model` remains generated externally. The finetuned model's attached training file labels its protocol `exploratory_leaked_pretraining`; do not call its evaluation leakage-free. If an old model lacks raw attempts, the evaluation should show raw validity as unavailable, not invent it.

For a first validation run, set `SMOKE_TEST=True`, run from the beginning, then set `SMOKE_TEST=False` for the full run. Smoke outputs go to a separate folder.